In [ ]:
import os
import sys
import json
import re

import numpy as np
import pandas as pd
from collections import defaultdict

import matplotlib.pyplot as plt

import nltk
from nltk.tokenize import word_tokenize

In [ ]:
# Parent directory PATH
sys.path.append("..")

from utils.json import load_json, save_json
from data.dataset import json_keys, json_structure, extract_documents, extract_annotations, remove_punctuation, remove_punctuation_dataframe, group_labels

In [ ]:
# Load the training data
file = "training.json"
path = "../data/raw/"
data = load_json(os.path.join(path, file))

In [ ]:
print(f"Type of loaded data: {type(data)}")
print(f"Number of documents: {len(data)}")

In [ ]:
# Explore your JSON data
json_structure(data, max_depth=10)

In [ ]:
# Get the JSON keys structure
structure = json_keys(data)

display(structure)

In [ ]:
# Insights using the first document
record_first = data[0]

if isinstance(data, list):
    print(f"Type document: {type(data[0])}")
    print(f"Keys document: {structure['main_keys']}")
    print()

    for main_keys in record_first:
        print(f"Key: {main_keys}")
        print(f"Type: {type(record_first[main_keys])}")
        print(f"Length: {len(record_first[main_keys])}")
        print(f"Value: {record_first[main_keys]}")
        print()

        if "data" in main_keys:
            for k in record_first["data"].keys():
                print(f"\tKey: {k}")
                print(f"\tType: {type(record_first['data'][k])}")
                print(f"\tLength: {len(record_first['data'][k])}")
                print(f"\tValue: {record_first['data'][k]}")
                print()

In [ ]:
extract_documents(data)

In [ ]:
# Extract all annotations
df_predictions = extract_annotations(data)

print(f"Extracted {len(df_predictions)} annotations")
display(df_predictions)

In [ ]:
annotation_counts = df_predictions.groupby("doc_id").size().sort_values(ascending=False)

print()
print("Top 5 documents by annotation count:")
for doc_id, count in annotation_counts.head(10).items():
    print(f"Document {doc_id} has {count} annotations")

In [ ]:
# Duplicated documents
true_counts = {}

# Count each unique document ID only once
for doc in data:
    doc_id = doc["data"]["id"]
    if doc_id in true_counts:
        true_counts[doc_id] += 1
    else:
        true_counts[doc_id] = 1

# Find actually duplicated IDs
actual_duplicates = {id: count for id, count in true_counts.items() if count > 1}

print(f"Number of documents: {len(data)}")
print(f"Number of unique document IDs: {len(true_counts)}")
print(f"Number of truly duplicated IDs: {len(actual_duplicates)}")
print()

if actual_duplicates:
    print("True duplicate document IDs:")
    for doc_id, count in actual_duplicates.items():
        print(f"ID {doc_id} appears {count} times")
else:
    print("No duplicate document IDs found")

In [ ]:
# Count the occurrences of each label
label_counts = df_predictions["label"].value_counts()

print("Label Distribution:")
for label, count in label_counts.items():
    print(f"{label}: {count} ({count/len(df_predictions)*100:.2f}%)")

labels = label_counts.index
counts = label_counts.values

# Plot the distribution using Matplotlib directly
plt.figure(figsize=(12, 7))
plt.bar(labels, counts, alpha=0.75)
plt.title("Distribution of Label Types")
plt.xlabel("Count")
plt.ylabel("Label")
plt.tight_layout()
plt.show()

In [ ]:
# we remove the punctuation from the scopes and add a new column to the dataframe with it
df_predictions = remove_punctuation_dataframe(df_predictions)

In [ ]:
# Common Negation Cues
neg_cues = df_predictions[df_predictions["label"] == "NEG"]["clean_text"].value_counts()
print("TOP Negation Cues: \n")
for cue, count in neg_cues.items():
    print(f"{cue}: {count}")

In [ ]:
# Common Uncertainty Cues
unc_cues = df_predictions[df_predictions["label"] == "UNC"]["clean_text"].value_counts()
print("TOP Uncertainty Cues: \n")
for cue, count in unc_cues.items():
    print(f"{cue}: {count}")

In [ ]:
# Calculate scope lengths
df_predictions["text_length"] = df_predictions["text"].apply(len)

scope_labels = ["NSCO", "USCO"]

# Average Scope's 
for label in scope_labels:
    avg_len = df_predictions[df_predictions["label"] == label]["text_length"].mean()
    print(f"Average {label} length: {avg_len:.2f} characters")

# Plot scope length distributions by label type
plt.figure(figsize=(12, 7))
for label in scope_labels:
    scope_lengths = df_predictions[df_predictions["label"] == label]["text_length"]
    plt.hist(scope_lengths, bins=50, alpha=0.75, label=label)

plt.title("Distribution of Scope Lengths")
plt.xlabel("Length in Characters")
plt.ylabel("Frequency")
plt.legend()
plt.xlim(0, 100) # Limit to most common
plt.tight_layout()
plt.show()

### Create groups of Cues-Scope
We agrupate them based on distances in the text. A scope is assigned to the closest cue.

In [ ]:
df_predictions = group_labels(df_predictions)

In [ ]:
df_predictions.head()

In [ ]:
# Get negation and uncertainty cues
negation_esp = pd.read_csv("../data/lexicons/negation/negation_esp.csv")
negation_cat = pd.read_csv("../data/lexicons/negation/negation_cat.csv")
uncertainty_esp = pd.read_csv("../data/lexicons/uncertainty/uncertainty_esp.csv")
uncertainty_cat = pd.read_csv("../data/lexicons/uncertainty/uncertainty_cat.csv")

# Combine both
negation_cues = pd.concat([negation_esp, negation_cat])
uncertainty_cues = pd.concat([uncertainty_esp, uncertainty_cat])

# Remove duplicated cues
negation_cues = negation_cues.drop_duplicates()
uncertainty_cues = uncertainty_cues.drop_duplicates()

In [ ]:
def preprocess_text(text):
    """Preprocess the text for rule-based analysis"""

    # it always starts in "motiu d'ingres" so just take it from there

    # Usar expresiones regulares para capturar lo que hay entre "hola" y "adiós"
    result = re.search(r"motiu d'ingres(.*)destinacio a l'alta", text)

    if result:
        captured_text = result.group(1).strip()
        # print(result.group(1).strip())  # Resultado sin espacios extra
    else:
        raise Exception ("No se encontró texto entre 'hola' y 'hoy'")

    
    # Handle special characters and standardize text
    final_text = captured_text.lower()

    # Add other preprocessing steps as needed

    return final_text

def tokenize_text(text):
    """Split text into tokens (words, punctuation)"""
    # You can use spaCy, NLTK, or custom tokenization
    import re
    tokens = re.findall(r'\w+|[^\w\s]', text)
    return tokens

# def find_cues(tokens, cue_lexicon):
#     """Find all instances of cues in the text"""
#     cues = []
#     cue_terms = cue_lexicon['term']
#     for i, token in enumerate(tokens):
#         if token in cue_terms.values:
#             cues.append({"index": i, "token": token})
#     return cues

def find_cues(tokens, cue_lexicon, max_ngram=2):
    """Find cues using n-gram approach, avoiding duplicates and overlaps."""
    cues = []
    cue_terms = cue_lexicon['term']
    text = ' '.join(tokens)
    sorted_cues = sorted(cue_terms, key=len, reverse=True)

    matched_indices = set()  # Keep track of token indices that have been matched

    for cue in sorted_cues:
        cue_tokens = cue.split()
        cue_length = len(cue_tokens)

        if cue_length > max_ngram:
            continue

        for i in range(len(tokens) - cue_length + 1):
            # Skip if the current token index has already been matched
            if i in matched_indices:
                continue

            # Check if the current cue matches the tokens at this position
            if tokens[i:i+cue_length] == cue_tokens:
                # Add the cue to the list of found cues
                cues.append({
                    "index": i,
                    "token": cue,
                    "length": cue_length
                })

                # Mark all token indices covered by this cue as matched
                for j in range(i, i+cue_length):
                    matched_indices.add(j)

                # Break to avoid detecting the same cue multiple times at the same position
                break
    return cues


def determine_scope(tokens, cue_index, window_size=5, direction="forward"):
    """Determine the scope of a cue based on linguistic rules"""
    # Start with basic window
    if direction == "forward":
        start = cue_index + 1
        end = min(cue_index + window_size + 1, len(tokens))
    else:
        start = max(0, cue_index - window_size)
        end = cue_index
    
    # Refine scope based on punctuation
    for i in range(start, end):
        if tokens[i] in ['.', ',', ';', ':', '!', '?']:
            end = i
            break
    
    return tokens[start:end]

def apply_negex(text, negation_cues, window_size=5):
    """Apply NegEx algorithm to find negation cues and their scopes"""
    processed_text = preprocess_text(text)
    tokens = tokenize_text(processed_text)
    results = []
    
    # Find all negation cues
    cues = find_cues(tokens, negation_cues)
    
    # For each cue, determine its scope
    for cue in cues:
        # Default to looking forward for scope
        scope_tokens = determine_scope(tokens, cue["index"], window_size)
        
        # Store the result
        results.append({
            "cue": cue["token"],
            "cue_position": cue["index"],
            "scope": " ".join(scope_tokens),
            "scope_start": cue["index"] + 1,
            "scope_end": cue["index"] + len(scope_tokens) + 1
        })
    
    return results

def apply_uncertainty(text, uncertainty_cues, window_size=5):
    """Apply NegEx algorithm to find negation cues and their scopes"""
    processed_text = preprocess_text(text)
    tokens = tokenize_text(processed_text)
    
    results = []
    
    # Find all negation cues
    cues = find_cues(tokens, uncertainty_cues)
    
    # For each cue, determine its scope
    for cue in cues:
        # Default to looking forward for scope
        scope_tokens = determine_scope(tokens, cue["index"], window_size)
        
        # Store the result
        results.append({
            "cue": cue["token"],
            "cue_position": cue["index"],
            "scope": " ".join(scope_tokens),
            "scope_start": cue["index"] + 1,
            "scope_end": cue["index"] + len(scope_tokens) + 1
        })
    
    return results

def apply_linguistic_rules(tokens, cue_index):
    """Apply linguistic rules to refine scope detection"""
    # Example rule: Scope extends until the next punctuation
    scope_end = cue_index + 1
    while scope_end < len(tokens) and tokens[scope_end] not in ['.', ',', ';', ':', '!', '?']:
        scope_end += 1
    
    return scope_end


def evaluate_system(predictions, gold_standard):
    """Evaluate the system against gold standard annotations"""
    # Calculate precision, recall, F1 for cue detection
    # cue_precision = 
    # Calculate precision, recall, F1 for scope detection
    # scope_precision = 
    # Return evaluation metrics
    return {
        # "cue_precision": cue_precision,
        # "cue_recall": cue_recall,
        # "cue_f1": cue_f1,
        # "scope_precision": scope_precision,
        # "scope_recall": scope_recall,
        # "scope_f1": scope_f1
    }
    
def process_dataset(data):
    """Process the entire dataset with the rule-based system"""
    results = []
    
    for doc in data:
        text = doc["data"]["text"]
        
        # Apply NegEx for negation
        negation_results = apply_negex(text, negation_cues)
        
        # Apply similar algorithm for uncertainty
        uncertainty_results = apply_uncertainty(text, uncertainty_cues)
        
        # Combine results
        doc_results = {
            "doc_id": doc["data"]["id"],
            "negation_results": negation_results,
            "uncertainty_results": uncertainty_results
        }
        
        results.append(doc_results)
    
    return results

In [ ]:
raw_results = process_dataset(data)
raw_results[0]

In [ ]:
# Process each document (this can probably more compressed)
def format_raw_results(results):
    # Create a list to hold all rows for the dataframe
    rows = []
    result_id_counter = 0


    for doc_index, doc in enumerate(results):
        doc_id = doc['doc_id']
        
        # Process negation results
        for neg_result in doc['negation_results']:
            cue = neg_result['cue']
            cue_position = neg_result['cue_position']
            scope = neg_result['scope']
            scope_start = neg_result['scope_start']
            scope_end = neg_result['scope_end']
            
            
            # Clean text
            #TODO don't know what is this meant for, I just focus on same output
            
            # If there is scope
            if scope:
                rows.append({
                'doc_index': doc_index,
                'doc_id': doc_id,
                'result_id': "ent" + str(result_id_counter),
                'start': scope_start,
                'end': scope_end,
                'label': 'NSCO',
                'text': scope,
                'clean_text': cue,
                'text_length': len(scope),
                'scope': scope
            })
            
            # There is always a cue            
            rows.append({
                'doc_index': doc_index,
                'doc_id': doc_id,
                'result_id': "ent" + str(result_id_counter),
                'start': cue_position,
                'end': cue_position + len(cue),
                'label': 'NEG',
                'text': cue,
                'clean_text': None,
                'text_length': len(cue),
                'scope': scope
            })
                
            result_id_counter += 1
        
        # Process uncertainty results (if any)
        for unc_result in doc['uncertainty_results']:
            cue = unc_result['cue']
            cue_position = unc_result['cue_position']
            scope = unc_result['scope']
            scope_start = unc_result['scope_start']
            scope_end = unc_result['scope_end']
            
            
            # Clean text
            #TODO don't know what is this meant for, I just focus on same output
            
            # If there is scope
            if scope:
                rows.append({
                'doc_index': doc_index,
                'doc_id': doc_id,
                'result_id': "ent" + str(result_id_counter),
                'start': scope_start,
                'end': scope_end,
                'label': 'USCO',
                'text': scope,
                'clean_text': cue,
                'text_length': len(scope),
                'scope': scope
            })
            
            # There is always a cue            
            rows.append({
                'doc_index': doc_index,
                'doc_id': doc_id,
                'result_id': "ent" + str(result_id_counter),
                'start': cue_position,
                'end': cue_position + len(cue),
                'label': 'UNC',
                'text': cue,
                'clean_text': None,
                'text_length': len(cue),
                'scope': scope
            })
                
            result_id_counter += 1
            
    return pd.DataFrame(rows)
            
# Create the dataframe
clean_results = format_raw_results(raw_results)

# Display the first few rows of the dataframe
print(clean_results.head())

In [ ]:
# problemas solucionados:
# ignora los signos de puntuación (CORREGIDO) 
# los cues que son palabras compuestas. Sin claras -> negation. (CORREGIDO) 
# problema cuando el scope es sobre si mismo (afebril)?  (VA BIEN)


# Problemas:
# solo detecta hacia delante (cistoscopia que es negativa) vs (negativa para lesiones malignas que se)  --> Solución, congruencia con código de Yaira
# no sabe el fin del scope (creo que negex original también sabe las palabras clave (estenosis focales...) con lo que sabe mejor cuando acabar (?)) --> Solución, congruencia con código de Yaira
# cuenta la posición distintamente (y no está únicamente relacionado con que haya quitado las partes iniciales del texto, lo cuenta por tokens y no characters)
